# DB7 Exercise B: window, stride, ACC centering and failure diagnostics

Every configuration gets a complete, independent train/validation/test experiment. **There is no validation-based selection of a window or stride.** Validation selects the checkpoint epoch within each individual fit. Test results do not control training, gate thresholds, normalization or experiment selection.

## Experiment grid and assumptions

| Setting | Protocol |
|---|---|
| Data | E1, `restimulus` labels 1–17; rest excluded; all 22 subject folders |
| Splits | Within each subject and gesture: 4 training, 1 validation, 1 test repetition |
| Split seed | `42 + 1009*subject + 9176*gesture`; identical repetition membership across configurations |
| Window lengths | 200, then 400, then 600 ms; same length across train, validation and test |
| Training/validation strides | 50, 100, 200 ms; a full fit for each combination |
| CNN variants | Original ACC; ACC centered by subtracting each channel's mean inside the current window |
| Classifier reports | CNN; EMG feature classifier; frozen gate between them |
| Gate | Switch to EMG only if CNN confidence <0.70 and EMG confidence >0.80 |
| Refit / augmentation | No training+validation refit; no augmentation |
| Randomness | One fixed model seed per subject, reset before every CNN fit |
| Fit budget | 22 × 3 windows × 3 strides × 2 ACC variants = **396 full CNN fits** |

The 50/100/200 ms stride grid, a single seed and keeping the current architecture fixed are experiment assumptions. Centering and gating are measured separately. The EMG classifier is fitted once per subject/window/stride on training windows and reused for both ACC variants.

## Inputs and preprocessing

Input shape is `[batch, 48 channels, time]`: 12 EMG + 36 supplied ACC columns. At the aligned 2 kHz row grid, 200/400/600 ms contain 400/800/1200 samples. EMG gets order-4 zero-phase 20–450 Hz bandpass plus 50 Hz notch (Q=30), within each annotated active repetition. ACC uses the supplied aligned rows; the nominal ACC_FS constant does not establish its physical sampling rate. Trim 100 ms from each end and never cross a repetition boundary. Fit per-channel mean/SD only on each CNN's actual training windows, after optional ACC centering. Apply those saved values to validation and test.

This measures active-gesture, within-subject classification. Annotated repetition filtering uses future samples and is not continuous causal deployment. `stimulus` is saved for label-timing diagnostics; it is not the training target. Folder IDs are preserved; previous S18 internal identity and physical channel-order questions remain unresolved and are recorded in identity files.

## CNN and gate

The existing CNN is unchanged: three parallel multi-kernel stages (3/5/7 kernels), 64→128→256 learned channels, channel attention after each stage, two temporal max pools, temporal attention pooling, and a 256→128→17 head. Each parallel branch contains two convolutions with BatchNorm/ReLU. Dropout is 0.15.

Adam, LR 0.0003, weight decay 0.0001, cosine schedule, batch 128 and gradient clipping 5. Maximum 150 epochs, patience 15, earliest stopping at epoch 20; the saved minimum-validation-loss checkpoint may be earlier. Maximum epochs and stopping rule are fixed, but different strides produce different training counts and optimizer steps; both are saved.

The EMG classifier is StandardScaler + shrinkage LDA using 72 training-derived features: channel-wise RMS, MAV, mean absolute sample difference, zero-crossing fraction, mean frequency and median frequency. Their presence in this reference classifier does **not** establish that the CNN lacks these features. Gate confidences are uncalibrated; calibration and harmful switches are reported.

## Three clearly labeled evaluation grids

1. **Primary:** test stride matches training and validation stride; first endpoint is 100 ms trim + window length. This is the requested full matched train/validation/test experiment.
2. **Common:** score the same endpoints for all nine window/stride configurations, starting 700 ms after onset, every 200 ms. Use this for paired comparisons that avoid different timing coverage.
3. **Dense diagnostic:** save test predictions every 50 ms, then subsample to 50/100/200 ms with every available offset. This isolates output-frequency effects in the same fitted model; it is not a training-stride experiment.

## Reading the diagnostics

Each `w<ms>/s<ms>/S<id>/<variant>/test/` folder contains:

- `predictions.csv`: every dense-grid decision, true/predicted gesture, confidence, phase, source sample interval, native repetition, three grid masks, both-wrong flag, gate recovery/harm.
- `gesture_errors.csv`, `gesture_error_bars.png`: counts of right/wrong decisions for each gesture and grid; subject/window/stride are explicit in aggregate tables.
- `probabilities.npz`, `calibration_*.csv`, `confusion_*.csv`: all class probabilities and error structure for CNN, EMG and gate.
- `features_embeddings.npz`: 300 physical features plus the CNN's 256-dimensional pooled representation for every window. Includes 72 reference EMG features, ACC means/SD/differences, relative EMG activation, spectral fractions, raw signal quality indicators and within-window amplitude change.
- `feature_correct_wrong.csv`: correct-vs-wrong comparisons within the same subject/gesture, both overall and within early/middle/late phases. Differences use training feature SD for scale; these are descriptive, not independent-window significance tests.
- `training_similarity.csv`: nearest training examples in physical-feature and learned-representation spaces, distance to the true gesture and CNN-predicted gesture, and traceable nearest-example positions. True labels are used only to diagnose errors after prediction.
- `phase_errors.csv`, `repetition_failures.csv`, `output_stride_offsets.csv`: where errors occur, longest consecutive error runs, both-classifier failures, and output-stride sensitivity.
- `attention.npz`: learned feature-channel and temporal attention for each test window. Attention describes model behavior; it is not proof of causal sensor importance.
- `waveform_cases.npz` and index: highest-confidence correct and wrong example per gesture with raw EMG, filtered EMG, ACC, stimulus and restimulus. These are illustrative selected examples, not random samples. Sample intervals let you retrieve every other case from the original MAT file.

Train and validation predictions/features are saved too, enabling comparisons to test. Each fit saves weights, its training-only normalizer, per-epoch/per-gesture learning histories, timing, optimizer-step count, and input/BatchNorm activation shift audits. BatchNorm audits do not update running statistics. Subject folders save repetition membership, input hash, identity and stimulus/restimulus intervals. `centering_paired_recovery.csv` records which identical decisions centering fixes or damages.

Root `all_subject_metrics.csv`, `all_gesture_errors.csv`, mean-subject metrics and subject-by-gesture error heatmaps make results comparable. Check `completion.json` and absence of failure files before accepting a run as complete. ZIPs are written even on ordinary Python exceptions; a forced platform termination may prevent final archiving.

## Execution and interpretation

Attach Kaggle dataset **rayaanraza1/ninapro-db7**, enable GPU, and run all cells. No package installation or previous kernel outputs are required inside the notebook. GitHub runs an 18-fit S1/two-epoch preflight covering every combination, then nine full jobs with 44 CNN fits each, ordered by window size. Smoke scores are not research results. Manual full-grid execution may exceed a single Kaggle session; set the window/stride lists to one combination to match a full GitHub job.

These test repetitions have been examined before. This study is exploratory and cannot establish a new independently confirmed 97% accuracy. Similarities/differences suggest hypotheses; later controlled ablations are needed to show what actually causes errors. No correction is inferred from one feature difference alone, and repeated extrema are indicators rather than proof of ADC clipping.


## 1. Imports and fixed experiment configuration

In [ ]:
import os, gc, time, json, warnings, random, re, shutil
from pathlib import Path
from copy import deepcopy
from math import gcd
import numpy as np
import pandas as pd
from scipy import io
from scipy.signal import resample_poly, butter, sosfiltfilt, iirnotch, filtfilt
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
warnings.filterwarnings('default')
import matplotlib
matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

import hashlib


In [ ]:
def _find_kaggle_input() -> Path:
    base = Path('/kaggle/input')
    if not base.exists():
        return Path('/kaggle/input/ninapro-db7/Dataset')

    def _has_subjects(p):
        return p.is_dir() and any((c.is_dir() and c.name.lower().startswith('subject_') for c in p.iterdir()))

    def _search(root, depth=0):
        if depth > 5:
            return None
        if _has_subjects(root):
            return root
        try:
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    result = _search(child, depth + 1)
                    if result is not None:
                        return result
        except PermissionError:
            pass
        return None
    result = _search(base)
    return result if result else Path('/kaggle/input/ninapro-db7/Dataset')

class Config:
    KAGGLE_INPUT = _find_kaggle_input()
    KAGGLE_WORKING = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd() / 'eda_working'
    EXERCISE_IDS = (1,)
    GESTURE_MIN, GESTURE_MAX, N_CLASSES = (1, 17, 17)
    SUBJECTS = list(range(1, 23))
    RUN_SUBJECTS = SUBJECTS.copy()
    INTACT_SUBJECTS = list(range(1, 21))
    AMPUTEE_SUBJECTS = [21, 22]
    SEED = 42
    MODEL_SEED = 42
    EMG_FS, ACC_FS, TARGET_FS = (2000, 128, 2000)
    EMG_KEY, ACC_KEY, LBL_KEY = ('emg', 'acc', 'restimulus')
    N_EMG_CH = 12
    USE_ACC = True
    REPS_PER_GESTURE, TRAIN_REPS, VAL_REPS, TEST_REPS = (6, 4, 1, 1)
    BANDPASS_LOW_HZ, BANDPASS_HIGH_HZ, FILTER_ORDER = (20.0, 450.0, 4)
    NOTCH_HZ, NOTCH_Q = (50.0, 30.0)
    WIN_MS, STEP_MS, TRIM_MS = (400, 100, 100)
    WIN_SAMPLES, STEP_SAMPLES, TRIM_SAMPLES = (800, 200, 200)
    DROPOUT, LR, WEIGHT_DECAY, GRAD_CLIP = (0.15, 0.0003, 0.0001, 5.0)
    MIN_EPOCHS, MAX_EPOCHS, PATIENCE, BATCH_SIZE = (20, 150, 15, 128)
    NUM_WORKERS = 0
    USE_ADAMW, AUGMENT_TRAIN, EVALUATE_TEST, REFIT_ON_TRAIN_PLUS_VAL = (False, False, False, False)
    LABEL_SMOOTHING = 0.0
    EMG_GAIN_STD, EMG_NOISE_STD = (0.1, 0.01)
    RUN_CNN = True
    RAW_CASES_PER_SUBJECT = 2
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AUTOMATION = {}
DIAG_RAW_AUDIT = []
Config.KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
print('Exercise B: every window and stride receives independent train/validation/test evaluation.')


In [ ]:
"""Controlled window/stride experiments and traceable within-subject failures."""
import joblib
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import NearestNeighbors
from threadpoolctl import threadpool_limits



In [ ]:
Config.WINDOWS_MS = [200, 400, 600]
Config.STRIDES_MS = [50, 100, 200]
Config.SMOKE = False
Config.GATE_CNN_THRESHOLD = 0.70
Config.GATE_EMG_THRESHOLD = 0.80
Config.EVALUATION_GRID_MS = 50
Config.REFIT_ON_TRAIN_PLUS_VAL = False



## 2. Recording loading and repetition-local filtering

In [ ]:
class RawEMGACCPreprocessor:
    """
    Load RAW EMG, RAW accelerometer and labels from one NinaPro file.

    No filtering, no ACC resampling, no label masking and no windowing happen here.
    Keeping each source file separate preserves the exact temporal mapping needed to
    extract a matching ACC interval for every EMG repetition.
    """

    @staticmethod
    def _find_key(data, candidates):
        for candidate in candidates:
            for key in data:
                if key.lower() == candidate.lower():
                    return key
        return None

    @staticmethod
    def _time_major(array, expected_channels=None, name='signal'):
        array = np.asarray(array)
        if array.ndim == 1:
            array = array[:, None]
        if array.ndim != 2:
            raise RuntimeError(f'{name}: expected 2-D array, got shape {array.shape}.')
        if expected_channels is not None:
            if array.shape[1] == expected_channels:
                return array
            if array.shape[0] == expected_channels:
                return array.T
            raise RuntimeError(f'{name}: neither dimension matches expected {expected_channels} channels: {array.shape}.')
        if array.shape[0] < array.shape[1] and array.shape[0] <= 128:
            array = array.T
        return array

    def apply(self, mat_path: Path):
        data = io.loadmat(str(mat_path))
        emg_key = self._find_key(data, [Config.EMG_KEY])
        acc_key = self._find_key(data, [Config.ACC_KEY])
        lbl_key = self._find_key(data, [Config.LBL_KEY, 'stimulus', 'label', 'labels'])
        if emg_key is None:
            raise KeyError(f'No EMG key in {mat_path.name}.')
        if acc_key is None:
            raise KeyError(f'No ACC key in {mat_path.name}; EMG+ACC experiment requires accelerometer data.')
        if lbl_key is None:
            raise KeyError(f'No label key in {mat_path.name}.')
        emg = self._time_major(data[emg_key], expected_channels=Config.N_EMG_CH, name=f'{mat_path.name} EMG').astype(np.float32)
        acc = self._time_major(data[acc_key], expected_channels=None, name=f'{mat_path.name} ACC').astype(np.float32)
        labels = np.asarray(data[lbl_key]).reshape(-1).astype(np.int32)
        if len(emg) != len(labels):
            raise ValueError('EMG/label length mismatch; truncation is disabled')
        if lbl_key.lower() != 'restimulus':
            raise ValueError('Refined restimulus labels are required')
        n = len(emg)
        emg = emg[:n]
        labels = labels[:n]
        if len(acc) < 2:
            raise RuntimeError(f'{mat_path.name}: ACC has only {len(acc)} samples.')
        rep_key = self._find_key(data, ['rerepetition'])
        native = np.asarray(data[rep_key]).reshape(-1) if rep_key else None
        self.last_native_repetition = native[:n].astype(np.int32) if native is not None and len(native) >= n else None
        self.last_file_path = str(mat_path)
        unique_labels = np.unique(labels).astype(int).tolist()
        run_audit = []
        edges = np.r_[0, np.flatnonzero(np.diff(labels) != 0) + 1, len(labels)]
        for start, end in zip(edges[:-1], edges[1:]):
            gesture = int(labels[start])
            if not Config.GESTURE_MIN <= gesture <= Config.GESTURE_MAX:
                continue
            ids = np.unique(self.last_native_repetition[start:end]).astype(int).tolist() if self.last_native_repetition is not None else []
            run_audit.append(dict(gesture=gesture, start=int(start), end=int(end), duration_seconds=float((end - start) / Config.EMG_FS), native_rerepetition_ids=ids))
        ratio = len(acc) / float(len(emg))
        DIAG_RAW_AUDIT.append(dict(file=str(mat_path), emg_shape=list(emg.shape), acc_shape=list(acc.shape), original_emg_shape=list(np.shape(data[emg_key])), original_label_length=int(np.size(data[lbl_key])), label_key=lbl_key, labels_present=unique_labels, rerepetition_key=rep_key, rerepetition_length=int(len(native)) if native is not None else None, sample_count_ratio_acc_to_emg=ratio, implied_acc_rate_if_shared_duration=ratio * Config.EMG_FS, timing_interpretation='Equal sample counts: may already be aligned; timestamps unverified.' if len(acc) == len(emg) else 'Unequal lengths: length-ratio mapping assumes shared start/end times; unverified.', selected_gesture_runs=run_audit))
        return (emg, acc, labels)

class RepetitionEMGFilter:
    """Zero-phase EMG filtering applied to one already-assigned repetition only."""

    def __init__(self):
        nyquist = Config.EMG_FS / 2.0
        self.sos = butter(Config.FILTER_ORDER, [Config.BANDPASS_LOW_HZ / nyquist, Config.BANDPASS_HIGH_HZ / nyquist], btype='bandpass', output='sos')
        self.b_notch, self.a_notch = iirnotch(Config.NOTCH_HZ / nyquist, Config.NOTCH_Q)

    def apply(self, repetition_emg):
        repetition_emg = np.asarray(repetition_emg, dtype=np.float32)
        if repetition_emg.ndim != 2:
            raise ValueError(f'Expected (time,channels), got {repetition_emg.shape}.')
        if len(repetition_emg) < 64:
            raise RuntimeError(f'EMG repetition unexpectedly short: {len(repetition_emg)} samples.')
        filtered = sosfiltfilt(self.sos, repetition_emg, axis=0)
        filtered = filtfilt(self.b_notch, self.a_notch, filtered, axis=0)
        return filtered.astype(np.float32)

class RepetitionACCResampler:
    """
    Map an EMG repetition interval to the corresponding interval of the RAW ACC
    recording from the SAME source file, then resample only that ACC slice.

    This is leakage-safe because resample_poly never sees ACC samples belonging
    to another train/validation/test repetition.
    """

    @staticmethod
    def extract_matching_interval(file_acc: np.ndarray, file_emg_length: int, emg_start: int, emg_end: int) -> np.ndarray:
        if not 0 <= emg_start < emg_end <= file_emg_length:
            raise ValueError(f'Invalid EMG interval [{emg_start}, {emg_end}) for file length {file_emg_length}.')
        ratio = len(file_acc) / float(file_emg_length)
        acc_start = int(np.ceil(emg_start * ratio))
        acc_end = int(np.ceil(emg_end * ratio))
        acc_start = max(0, min(acc_start, len(file_acc) - 1))
        acc_end = max(acc_start + 1, min(acc_end, len(file_acc)))
        acc_rep = file_acc[acc_start:acc_end].copy()
        if len(acc_rep) < 2:
            raise RuntimeError(f'Mapped ACC repetition is too short: {len(acc_rep)} samples.')
        return acc_rep

    @staticmethod
    def resample_to_emg_length(acc_rep: np.ndarray, target_length: int) -> np.ndarray:
        source_length = len(acc_rep)
        divisor = gcd(source_length, target_length)
        up = target_length // divisor
        down = source_length // divisor
        resampled = resample_poly(acc_rep, up, down, axis=0).astype(np.float32)
        if len(resampled) > target_length:
            resampled = resampled[:target_length]
        elif len(resampled) < target_length:
            pad_count = target_length - len(resampled)
            pad = np.repeat(resampled[-1:, :], pad_count, axis=0)
            resampled = np.vstack([resampled, pad])
        if len(resampled) != target_length:
            raise RuntimeError(f'ACC resampling length mismatch: {len(resampled)} != {target_length}.')
        return resampled.astype(np.float32)
print('Raw EMG+ACC loader, repetition-local EMG filter and ACC resampler defined.')


In [ ]:
class SubjectLoader:
    """
    Disk-safe subject loader.

    Only ONE subject is held in memory at a time. No raw signal cache and no
    overlapping-window cache is written to /kaggle/working.
    """

    def __init__(self):
        self.preprocessor = RawEMGACCPreprocessor()
        self.rep_filter = RepetitionEMGFilter()
        self.acc_resampler = RepetitionACCResampler()

    def _find_subject_dir(self, sid: int) -> Path:
        candidates = [Config.KAGGLE_INPUT / f'Subject_{sid}', Config.KAGGLE_INPUT / f'subject_{sid}', Config.KAGGLE_INPUT / f'S{sid}', Config.KAGGLE_INPUT / f's{sid}']
        for path in candidates:
            if path.is_dir():
                return path
        for path in sorted(Config.KAGGLE_INPUT.iterdir()):
            if path.is_dir() and path.name.lower().endswith(str(sid)):
                return path
        raise FileNotFoundError(f'Cannot find subject {sid} under {Config.KAGGLE_INPUT}.')

    @staticmethod
    def _is_exercise_file(path: Path, exercise_id: int) -> bool:
        name = path.stem.upper()
        return bool(re.search(f'(^|_)E{exercise_id}(_|$)', name))

    def _selected_files(self, sid: int):
        subject_dir = self._find_subject_dir(sid)
        all_mat_files = sorted(subject_dir.glob('*.mat')) or sorted(subject_dir.rglob('*.mat'))
        selected = []
        for exercise_id in Config.EXERCISE_IDS:
            matches = [path for path in all_mat_files if self._is_exercise_file(path, exercise_id)]
            if not matches:
                raise FileNotFoundError(f'S{sid:02d}: missing E{exercise_id}. Available: {[p.name for p in all_mat_files[:15]]}')
            selected.extend(matches)
        return selected

    def _load_raw_parts_from_source(self, sid: int):
        """
        Read E1/E2 directly from the read-only Kaggle input and keep them only
        for the current subject.
        """
        parts = []
        acc_channel_counts = set()
        for index, mat_file in enumerate(self._selected_files(sid)):
            emg, acc, labels = self.preprocessor.apply(mat_file)
            if emg.shape[1] != Config.N_EMG_CH:
                raise RuntimeError(f'S{sid:02d}, {mat_file.name}: expected {Config.N_EMG_CH} EMG channels, got {emg.shape[1]}.')
            if acc.shape[1] <= 0:
                raise RuntimeError(f'S{sid:02d}, {mat_file.name}: ACC is required.')
            acc_channel_counts.add(int(acc.shape[1]))
            parts.append({'file_index': int(index), 'file_name': mat_file.name, 'emg': emg.astype(np.float32, copy=False), 'acc': acc.astype(np.float32, copy=False), 'labels': labels.astype(np.int32, copy=False), 'native_repetition': self.preprocessor.last_native_repetition, 'source_path': self.preprocessor.last_file_path})
        if len(acc_channel_counts) != 1:
            raise RuntimeError(f'S{sid:02d}: inconsistent ACC channel counts across E1/E2: {sorted(acc_channel_counts)}.')
        return (parts, int(next(iter(acc_channel_counts))))

    @staticmethod
    def _constant_label_runs(labels: np.ndarray):
        if len(labels) == 0:
            return
        boundaries = np.flatnonzero(np.diff(labels) != 0) + 1
        starts = np.r_[0, boundaries]
        ends = np.r_[boundaries, len(labels)]
        for start, end in zip(starts, ends):
            yield (int(start), int(end), int(labels[start]))

    @staticmethod
    def _repetition_id(sid, gesture, repetition_index):
        return (int(sid), int(gesture), int(repetition_index + 1))

    @staticmethod
    def _split_repetition_indices(sid, gesture):
        rng = np.random.default_rng(Config.SEED + 1009 * int(sid) + 9176 * int(gesture))
        perm = rng.permutation(Config.REPS_PER_GESTURE).tolist()
        test_idx = sorted(perm[:Config.TEST_REPS])
        val_idx = sorted(perm[Config.TEST_REPS:Config.TEST_REPS + Config.VAL_REPS])
        train_idx = sorted(perm[Config.TEST_REPS + Config.VAL_REPS:])
        if len(train_idx) != Config.TRAIN_REPS or len(val_idx) != Config.VAL_REPS or len(test_idx) != Config.TEST_REPS:
            raise RuntimeError('Unexpected repetition split size.')
        return (train_idx, val_idx, test_idx)

    def _collect_repetitions(self, parts):
        runs_by_gesture = {gesture: [] for gesture in range(Config.GESTURE_MIN, Config.GESTURE_MAX + 1)}
        for part_index, part in enumerate(parts):
            for run_start, run_end, gesture in self._constant_label_runs(part['labels']):
                if gesture in runs_by_gesture:
                    runs_by_gesture[gesture].append({'part_index': int(part_index), 'start': int(run_start), 'end': int(run_end)})
        return runs_by_gesture


## 3. Unchanged multi-kernel CNN

In [ ]:
import torch
import torch.nn as nn

class ParallelMultiKernelBlock(nn.Module):
    """
    True multi-kernel block: parallel branches with different kernel sizes
    (default 3, 5, 7) processed at the SAME depth and concatenated along the
    channel dimension, then merged with a 1x1 conv. This captures multi-scale
    temporal patterns simultaneously (unlike a sequential 7->5->3 design,
    which only changes kernel size across depth, not within one stage).
    """

    def __init__(self, in_ch, out_ch, kernels=(3, 5, 7), pool=True, dropout=0.1):
        super().__init__()
        for k in kernels:
            assert k % 2 == 1, f"kernel size {k} must be odd so that padding=kernel//2 gives symmetric 'same' padding"
        branch_sizes = self._split_channels(out_ch, len(kernels))
        self.branches = nn.ModuleList([nn.Sequential(nn.Conv1d(in_ch, b_ch, k, padding=k // 2, bias=False), nn.BatchNorm1d(b_ch), nn.ReLU(inplace=True), nn.Conv1d(b_ch, b_ch, k, padding=k // 2, bias=False), nn.BatchNorm1d(b_ch), nn.ReLU(inplace=True)) for k, b_ch in zip(kernels, branch_sizes)])
        self.merge = nn.Sequential(nn.Conv1d(out_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True))
        self.drop = nn.Dropout1d(dropout) if dropout > 0 else nn.Identity()
        self.pool = nn.MaxPool1d(2) if pool else nn.Identity()

    @staticmethod
    def _split_channels(total, n):
        base, rem = divmod(total, n)
        return [base + 1 if i < rem else base for i in range(n)]

    def forward(self, x):
        x = torch.cat([branch(x) for branch in self.branches], dim=1)
        x = self.merge(x)
        x = self.drop(x)
        return self.pool(x)

class ChannelAttentionBlock(nn.Module):
    """
    Squeeze-and-Excitation style CHANNEL attention. Learns which learned feature channels
    matter most; it does NOT attend across time
    steps. Named explicitly so it isn't confused with temporal attention.
    """

    def __init__(self, in_ch, reduction=4):
        super().__init__()
        reduced = max(in_ch // reduction, 1)
        self.attention = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Conv1d(in_ch, reduced, 1), nn.ReLU(inplace=True), nn.Conv1d(reduced, in_ch, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.attention(x)

class TemporalAttentionPool(nn.Module):
    """
    Learned attention-weighted pooling over the time axis, replacing plain
    global average pooling. A 1x1 conv scores every time step, softmax turns
    scores into weights, and the weighted sum replaces a uniform mean — so
    the model can down-weight uninformative (e.g. resting) time steps instead
    of averaging them in blindly.
    """

    def __init__(self, in_ch):
        super().__init__()
        self.score = nn.Conv1d(in_ch, 1, kernel_size=1)

    def forward(self, x):
        weights = torch.softmax(self.score(x), dim=-1)
        return (x * weights).sum(dim=-1)

class OriginalMultiKernelAttention1DCNN(nn.Module):

    def __init__(self, n_channels, n_classes, dropout=0.15):
        super().__init__()
        self.n_channels = int(n_channels)
        self.stage1 = ParallelMultiKernelBlock(n_channels, 64, kernels=(3, 5, 7), pool=True, dropout=dropout)
        self.attn1 = ChannelAttentionBlock(64)
        self.stage2 = ParallelMultiKernelBlock(64, 128, kernels=(3, 5, 7), pool=True, dropout=dropout)
        self.attn2 = ChannelAttentionBlock(128)
        self.stage3 = ParallelMultiKernelBlock(128, 256, kernels=(3, 5, 7), pool=False, dropout=dropout)
        self.attn3 = ChannelAttentionBlock(256)
        self.temporal_pool = TemporalAttentionPool(256)
        self.head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(128, n_classes))

    def forward(self, x):
        x = self.stage1(x)
        x = self.attn1(x)
        x = self.stage2(x)
        x = self.attn2(x)
        x = self.stage3(x)
        x = self.attn3(x)
        x = self.temporal_pool(x)
        return self.head(x)

    def count_params(self):
        return sum((parameter.numel() for parameter in self.parameters() if parameter.requires_grad))
import torch.nn.functional as F


## 4. Training, checkpoint selection and metrics

In [ ]:
class Trainer:

    def __init__(self, model, save_path: Path, n_classes: int):
        self.model = model.to(Config.DEVICE)
        self.save_path = save_path
        self.n_classes = n_classes
        self.history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
        self.best_epoch = 0
        self.best_val_loss = float('inf')
        self.train_wall = 0.0

    def _run_epoch(self, loader, optimizer=None, criterion=None):
        training = optimizer is not None
        self.model.train(training)
        total_loss, correct, total = (0.0, 0, 0)
        context = torch.enable_grad() if training else torch.no_grad()
        with context:
            for X, y in loader:
                X = X.to(Config.DEVICE, non_blocking=True)
                y = y.to(Config.DEVICE, non_blocking=True)
                if training and Config.AUGMENT_TRAIN:
                    X = X.clone()
                    emg = X[:, :Config.N_EMG_CH]
                    gain = torch.exp(Config.EMG_GAIN_STD * torch.randn(emg.shape[0], emg.shape[1], 1, device=emg.device))
                    X[:, :Config.N_EMG_CH] = emg * gain + Config.EMG_NOISE_STD * torch.randn_like(emg)
                logits = self.model(X)
                loss = criterion(logits, y)
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP)
                    optimizer.step()
                total_loss += loss.item() * len(y)
                correct += (logits.argmax(1) == y).sum().item()
                total += len(y)
        return (total_loss / max(total, 1), correct / max(total, 1))

    def fit(self, train_loader, val_loader):
        optimizer = (torch.optim.AdamW if Config.USE_ADAMW else Adam)(self.model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scheduler = CosineAnnealingLR(optimizer, T_max=Config.MAX_EPOCHS)
        patience_count = 0
        start_wall = time.perf_counter()
        for epoch in range(1, Config.MAX_EPOCHS + 1):
            tr_loss, tr_acc = self._run_epoch(train_loader, optimizer, criterion)
            vl_loss, vl_acc = self._run_epoch(val_loader, criterion=criterion)
            scheduler.step()
            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(vl_loss)
            self.history['train_acc'].append(tr_acc)
            self.history['val_acc'].append(vl_acc)
            if vl_loss < self.best_val_loss:
                self.best_val_loss = float(vl_loss)
                self.best_epoch = int(epoch)
                patience_count = 0
                torch.save(self.model.state_dict(), self.save_path)
            else:
                patience_count += 1
            if epoch == 1 or epoch % 10 == 0:
                print(f'  Epoch {epoch:3d} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}')
            if epoch >= Config.MIN_EPOCHS and patience_count >= Config.PATIENCE:
                print(f'  Early stop at epoch {epoch} (patience={Config.PATIENCE})')
                break
        self.train_wall = time.perf_counter() - start_wall
        print(f'  Selection training: {self.train_wall:.1f} s | best epoch={self.best_epoch} | best val_loss={self.best_val_loss:.4f}')
        return self

    def fit_fixed_epochs(self, train_loader, epochs: int):
        """Fresh final model training on train+validation after epoch selection."""
        optimizer = (torch.optim.AdamW if Config.USE_ADAMW else Adam)(self.model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scheduler = CosineAnnealingLR(optimizer, T_max=Config.MAX_EPOCHS)
        start_wall = time.perf_counter()
        for epoch in range(1, int(epochs) + 1):
            loss, acc = self._run_epoch(train_loader, optimizer, criterion)
            scheduler.step()
            if epoch == 1 or epoch % 10 == 0 or epoch == int(epochs):
                print(f'  Refit epoch {epoch:3d}/{int(epochs)} | loss={loss:.4f} | acc={acc:.4f}')
        self.train_wall = time.perf_counter() - start_wall
        torch.save(self.model.state_dict(), self.save_path)
        print(f'  Refit training: {self.train_wall:.1f} s')
        return self


In [ ]:
import sys, platform, traceback, zipfile
from datetime import datetime, timezone
from sklearn.metrics import classification_report, balanced_accuracy_score
from scipy.signal import welch

def json_write(path, obj):

    def convert(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, np.generic):
            return x.item()
        return str(x)
    Path(path).write_text(json.dumps(obj, indent=2, default=convert), encoding='utf-8')

def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)

def probability_metrics(y, p):
    """Uncalibrated confidence diagnostics; multiclass Brier is a sum over classes."""
    pred = p.argmax(1)
    confidence = p.max(1)
    correct = pred == y
    bins = np.minimum((confidence * 10).astype(int), 9)
    calibration = []
    ece = 0.0
    for b in range(10):
        mask = bins == b
        n = int(mask.sum())
        acc = float(correct[mask].mean()) if n else None
        conf = float(confidence[mask].mean()) if n else None
        calibration.append(dict(bin=b, lower=b / 10, upper=(b + 1) / 10, count=n, accuracy=acc, confidence=conf))
        if n:
            ece += n / len(y) * abs(acc - conf)
    targets = np.eye(p.shape[1])[y]
    result = dict(accuracy=float(correct.mean()), balanced_accuracy=float(balanced_accuracy_score(y, pred)), f1_macro=float(f1_score(y, pred, labels=np.arange(p.shape[1]), average='macro', zero_division=0)), f1_weighted=float(f1_score(y, pred, average='weighted', zero_division=0)), nll=float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()), brier_multiclass=float(((p - targets) ** 2).sum(1).mean()), ece_10_bins=float(ece), high_confidence_error_fraction=float(((confidence >= 0.9) & ~correct).mean()), n_windows=int(len(y)))
    return (result, calibration)

class DiagnosticTrainer(Trainer):
    """Preserves Trainer.fit selection logic; records each epoch without extra forwards."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.epoch_rows = []
        self.last_training = {}

    def _run_epoch(self, loader, optimizer=None, criterion=None):
        training = optimizer is not None
        self.model.train(training)
        total_loss, correct, total = (0.0, 0, 0)
        norms, pred_all, y_all = ([], [], [])
        tick = time.perf_counter()
        with torch.set_grad_enabled(training):
            for X, y in loader:
                X, y = (X.to(Config.DEVICE), y.to(Config.DEVICE))
                if training and Config.AUGMENT_TRAIN:
                    X = X.clone()
                    emg = X[:, :Config.N_EMG_CH]
                    gain = torch.exp(Config.EMG_GAIN_STD * torch.randn(emg.shape[0], emg.shape[1], 1, device=emg.device))
                    X[:, :Config.N_EMG_CH] = emg * gain + Config.EMG_NOISE_STD * torch.randn_like(emg)
                logits = self.model(X)
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Non-finite loss; inspect input arrays and normalizer')
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP)
                    if not torch.isfinite(norm):
                        raise FloatingPointError('Non-finite gradient norm')
                    norms.append(float(norm.item()))
                    optimizer.step()
                pred = logits.argmax(1)
                total_loss += loss.item() * len(y)
                correct += (pred == y).sum().item()
                total += len(y)
                pred_all.extend(pred.detach().cpu().tolist())
                y_all.extend(y.cpu().tolist())
        values = dict(loss=total_loss / total, accuracy=correct / total, f1_macro=float(f1_score(y_all, pred_all, labels=range(self.n_classes), average='macro', zero_division=0)), seconds=time.perf_counter() - tick)
        labels = np.asarray(y_all)
        guesses = np.asarray(pred_all)
        if not hasattr(self, 'class_epoch_rows'):
            self.class_epoch_rows = []
        for c in range(self.n_classes):
            selected = labels == c
            self.class_epoch_rows.append(dict(epoch=len(self.epoch_rows) + 1, split='train_online_dropout' if training else 'validation', gesture=c + Config.GESTURE_MIN, windows=int(selected.sum()), recall=float((guesses[selected] == c).mean())))
        pd.DataFrame(self.class_epoch_rows).to_csv(self.save_path.parent / 'class_learning_history.csv', index=False)
        if training:
            self.last_training = {'train_' + k: v for k, v in values.items()}
            self.last_training.update(lr=optimizer.param_groups[0]['lr'], gradient_norm_mean=float(np.mean(norms)), gradient_norm_max=float(np.max(norms)), gradient_clipped_fraction=float(np.mean(np.array(norms) > Config.GRAD_CLIP)))
        else:
            row = dict(epoch=len(self.epoch_rows) + 1, **self.last_training, **{'val_' + k: v for k, v in values.items()})
            self.epoch_rows.append(row)
            pd.DataFrame(self.epoch_rows).to_csv(self.save_path.parent / 'history.csv', index=False)
        return (values['loss'], values['accuracy'])

def learning_report(trainer, directory):
    h = pd.DataFrame(trainer.epoch_rows)
    h.to_csv(directory / 'history.csv', index=False)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for split in ('train', 'val'):
        axes[0, 0].plot(h.epoch, h[split + '_loss'], label=split)
        axes[0, 1].plot(h.epoch, h[split + '_accuracy'], label=split)
    axes[0, 0].set_ylabel('Cross entropy')
    axes[0, 1].set_ylabel('Accuracy')
    axes[1, 0].plot(h.epoch, h.lr)
    axes[1, 0].set_ylabel('Learning rate')
    axes[1, 1].plot(h.epoch, h.gradient_norm_mean, label='Mean before clipping')
    axes[1, 1].plot(h.epoch, h.gradient_norm_max, label='Max before clipping')
    axes[1, 1].axhline(Config.GRAD_CLIP, color='gray', linestyle='--')
    axes[1, 1].set_ylabel('Gradient norm')
    for ax in axes.flat:
        ax.axvline(trainer.best_epoch, color='red', linestyle=':', label='Selected epoch')
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
    fig.suptitle('Online training uses dropout; checkpoint gap is measured separately in eval mode')
    save_figure(fig, directory / 'learning_curves.png')


## 5. Input and BatchNorm activation audit

In [ ]:
import zipfile, hashlib

def activation_audit(model, datasets, path):
    model.eval()
    bns = {n: m for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d)}
    before = {n: (m.running_mean.clone(), m.running_var.clone(), m.num_batches_tracked.clone()) for n, m in bns.items()}
    rows = []
    for split, d in datasets.items():
        agg = {}
        handles = []

        def hook(name):

            def fn(module, args):
                a = args[0].detach().float()
                axes = (0, 2)
                s = a.sum(axes).cpu().numpy().astype(np.float64)
                q = (a * a).sum(axes).cpu().numpy().astype(np.float64)
                n = a.shape[0] * a.shape[2]
                if name not in agg:
                    agg[name] = [s, q, n]
                else:
                    agg[name][0] += s
                    agg[name][1] += q
                    agg[name][2] += n
            return fn
        for name, m in bns.items():
            handles.append(m.register_forward_pre_hook(hook(name)))
        handles.append(model.register_forward_pre_hook(hook('__input__')))
        ids = np.unique(np.linspace(0, len(d) - 1, min(512, len(d)), dtype=int))
        try:
            with torch.no_grad():
                for start in range(0, len(ids), 64):
                    model(torch.stack([d[int(i)][0] for i in ids[start:start + 64]]).to(Config.DEVICE))
        finally:
            for handle in handles:
                handle.remove()
        for name, (s, q, n) in agg.items():
            mu = s / n
            sd = np.sqrt(np.maximum(q / n - mu * mu, 0))
            module = bns.get(name)
            rm = module.running_mean.cpu().numpy() if module is not None else np.zeros(len(mu))
            rs = np.sqrt(module.running_var.cpu().numpy() + module.eps) if module is not None else np.ones(len(mu))
            for c in range(len(mu)):
                rows.append(dict(split=split, layer=name, channel=c, sampled_windows=len(ids), mean=float(mu[c]), std=float(sd[c]), running_mean=float(rm[c]), running_std=float(rs[c]), mean_shift_running_sd=float((mu[c] - rm[c]) / rs[c]), std_ratio=float(sd[c] / rs[c])))
    for n, m in bns.items():
        assert all((torch.equal(a, b) for a, b in zip(before[n], (m.running_mean, m.running_var, m.num_batches_tracked))))
    frame = pd.DataFrame(rows)
    frame.to_csv(path / 'activation_distributions.csv', index=False)
    reference = frame[frame.split == 'train'][['layer', 'channel', 'mean', 'std']].rename(columns={'mean': 'training_observed_mean', 'std': 'training_observed_std'})
    delta = frame.merge(reference, on=['layer', 'channel'], validate='many_to_one')
    delta['shift_observed_training_sd'] = (delta['mean'] - delta.training_observed_mean) / (delta.training_observed_std + 1e-08)
    delta.to_csv(path / 'activation_training_comparison.csv', index=False)
    delta.assign(abs_shift=lambda d: d.shift_observed_training_sd.abs()).groupby(['split', 'layer']).agg(median_abs_shift=('abs_shift', 'median'), max_abs_shift=('abs_shift', 'max'), median_std_ratio_to_running=('std_ratio', 'median')).to_csv(path / 'activation_layer_summary.csv')
    json_write(path / 'activation_audit_manifest.json', dict(eval_mode=True, running_statistics_unchanged=True, max_windows_per_split=512, selection='Evenly spaced deterministic window indices; no fitting on held-out data'))


## 6. Traceable windows and training-only normalization

In [ ]:
def load_diagnostic_subject(subject, output):
    """Read one recording; preserve raw signals, labels and repetition provenance."""
    loader = SubjectLoader()
    parts, acc_channels = loader._load_raw_parts_from_source(subject)
    assert len(parts) == 1 and acc_channels == 36
    part = parts[0]
    raw_emg, acc, labels = part['emg'], part['acc'], part['labels']
    assert len(raw_emg) == len(acc) == len(labels)
    assert np.isfinite(raw_emg).all() and np.isfinite(acc).all()
    assert set(np.unique(labels)) == set(range(18))
    extras = io.loadmat(loader._selected_files(subject)[0], variable_names=['stimulus', 'subject', 'exercise'])
    stimulus = np.asarray(extras['stimulus']).reshape(-1)
    assert len(stimulus) == len(labels)
    rows = []
    signal = np.empty((len(labels), 48), dtype=np.float32)
    signal[:, :12] = 0  # Rest is never sampled by WindowDataset.
    signal[:, 12:] = acc
    for gesture, repetitions in loader._collect_repetitions(parts).items():
        assert len(repetitions) == 6
        train_ids, val_ids, test_ids = loader._split_repetition_indices(subject, gesture)
        native_seen = set()
        for index, repetition in enumerate(repetitions):
            start, end = repetition['start'], repetition['end']
            native = np.unique(part['native_repetition'][start:end])
            assert len(native) == 1 and 1 <= int(native[0]) <= 6 and int(native[0]) not in native_seen
            native_seen.add(int(native[0]))
            split = 'train' if index in train_ids else 'validation' if index in val_ids else 'test'
            signal[start:end, :12] = loader.rep_filter.apply(raw_emg[start:end])
            rows.append(dict(subject=subject, gesture=gesture, native_repetition=int(native[0]),
                             split=split, file_name=part['file_name'],
                             run_start=start, run_end=end))
    metadata = pd.DataFrame(rows)
    metadata.to_csv(output/'repetition_inventory.csv', index=False)
    json_write(output/'identity.json', dict(folder_subject=subject,
               internal_subject=int(extras['subject'].item()), exercise=int(extras['exercise'].item()),
               signal_shape=list(signal.shape), label_key=Config.LBL_KEY,
               raw_emg_sha256=hashlib.sha256(raw_emg.tobytes()).hexdigest()))
    boundaries = np.unique(np.r_[0, np.flatnonzero(np.diff(labels))+1,
                                 np.flatnonzero(np.diff(stimulus))+1, len(labels)])
    pd.DataFrame(dict(start=boundaries[:-1], end=boundaries[1:],
                     stimulus=stimulus[boundaries[:-1]], restimulus=labels[boundaries[:-1]])
                ).to_csv(output/'stimulus_restimulus_intervals.csv', index=False)
    return dict(signal=signal, raw_emg=raw_emg, stimulus=stimulus,
                restimulus=labels, metadata=metadata)


class WindowDataset(Dataset):
    """One subject, split and window/stride configuration; no split crosses a repetition."""
    def __init__(self, recording, split, window_ms, stride_ms, center_acc=False):
        self.recording = recording
        self.window_ms = int(window_ms)
        self.stride_ms = int(stride_ms)
        self.samples = 2 * self.window_ms
        self.center_acc = bool(center_acc)
        self.mean = np.zeros((48, 1), np.float32)
        self.std = np.ones((48, 1), np.float32)
        rows = []
        for repetition in recording['metadata'].to_dict('records'):
            if repetition['split'] != split:
                continue
            first_end = repetition['run_start'] + 200 + self.samples
            last_end = repetition['run_end'] - 200
            assert first_end <= last_end
            for end in range(first_end, last_end + 1, 2 * self.stride_ms):
                start = end - self.samples
                rows.append(dict(**repetition, window_start=start, window_end=end,
                    window_ms=self.window_ms, grid_stride_ms=self.stride_ms,
                    endpoint_ms=(end-repetition['run_start'])/2,
                    phase_fraction=((start+end)/2-repetition['run_start']) /
                                   (repetition['run_end']-repetition['run_start']),
                    stimulus_disagreement_fraction=float(np.mean(
                        recording['stimulus'][start:end] != repetition['gesture']))))
        self.meta = pd.DataFrame(rows)
        self.meta['phase'] = pd.cut(self.meta.phase_fraction, [0, 1/3, 2/3, 1],
                                    labels=['early', 'middle', 'late'], include_lowest=True).astype(str)
        self.y = self.meta.gesture.to_numpy(dtype=np.int64) - 1
        self.starts = self.meta.window_start.to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.y)

    def physical(self, index):
        start = self.starts[index]
        return self.recording['signal'][start:start+self.samples].T.copy()

    def raw(self, index):
        signal = self.physical(index)
        if self.center_acc:
            signal[12:] -= signal[12:].mean(axis=1, keepdims=True)
        return signal

    def __getitem__(self, index):
        signal = ((self.raw(index)-self.mean)/self.std).astype(np.float32)
        return torch.from_numpy(signal), int(self.y[index])


def fit_window_normalizer(training):
    """Fit only on actual training input windows, after optional ACC centering."""
    total = np.zeros(48, np.float64)
    squares = np.zeros(48, np.float64)
    count = 0
    for begin in range(0, len(training), 64):
        signal = np.stack([training.raw(i) for i in range(begin, min(begin+64, len(training)))]).astype(np.float64)
        total += signal.sum(axis=(0, 2))
        squares += np.square(signal).sum(axis=(0, 2))
        count += signal.shape[0]*signal.shape[2]
    mean = total/count
    std = np.sqrt(np.maximum(squares/count-mean*mean, 0))+1e-8
    return mean.astype(np.float32)[:, None], std.astype(np.float32)[:, None]



## 7. Physical signal features and classical EMG classifier

In [ ]:
def diagnostic_features(dataset):
    """Keep the first 72 features identical to the previous EMG gate classifier."""
    blocks = []
    for begin in range(0, len(dataset), 64):
        ids = range(begin, min(begin+64, len(dataset)))
        signal = np.stack([dataset.physical(i) for i in ids]).astype(np.float64)
        emg, acc = signal[:, :12], signal[:, 12:]
        difference = np.diff(emg, axis=-1)
        power = np.abs(np.fft.rfft(emg, axis=-1))**2
        frequency = np.fft.rfftfreq(emg.shape[-1], 1/Config.EMG_FS)
        total_power = power.sum(-1)+1e-30
        rms = np.sqrt(np.mean(emg**2, -1))
        features = [rms, np.mean(abs(emg), -1), np.mean(abs(difference), -1),
            np.mean(emg[:, :, :-1]*emg[:, :, 1:] < 0, -1),
            (power*frequency).sum(-1)/total_power,
            frequency[np.argmax(np.cumsum(power, axis=-1) >= total_power[:, :, None]/2, axis=-1)],
            acc.mean(-1), acc.std(-1), np.mean(abs(np.diff(acc, axis=-1)), -1),
            rms/(rms.sum(1, keepdims=True)+1e-30),
            np.max(abs(emg), -1)/(rms+1e-30),
            power[:, :, (frequency >= 20)&(frequency < 60)].sum(-1)/total_power,
            power[:, :, (frequency >= 250)&(frequency <= 450)].sum(-1)/total_power]
        raw = np.stack([dataset.recording['raw_emg'][dataset.starts[i]:dataset.starts[i]+dataset.samples].T for i in ids]).astype(np.float64)
        raw_rms = np.sqrt(np.mean(raw**2, -1))
        features.extend([raw_rms, np.max(abs(raw), -1),
                         np.mean(np.diff(raw, axis=-1) == 0, -1),
                         np.mean(raw == raw.max(-1, keepdims=True), -1),
                         np.sqrt(np.mean(emg[:, :, :emg.shape[-1]//2]**2, -1))/(rms+1e-30),
                         np.sqrt(np.mean(emg[:, :, emg.shape[-1]//2:]**2, -1))/(rms+1e-30)])
        blocks.append(np.concatenate(features, axis=1).astype(np.float32))
    emg_names = ['rms', 'mav', 'wl_mean', 'zc_fraction', 'mean_frequency', 'median_frequency']
    names = [f'emg_{name}_{ch+1:02}' for name in emg_names for ch in range(12)]
    names += [f'acc_{name}_{ch+1:02}' for name in ['mean', 'std', 'mean_abs_diff'] for ch in range(36)]
    extras = ['relative_rms', 'crest_factor', 'power_20_60_fraction', 'power_250_450_fraction',
              'raw_rms', 'raw_abs_peak', 'raw_flat_fraction', 'raw_max_repeat_fraction',
              'first_half_rms_ratio', 'second_half_rms_ratio']
    names += [f'emg_{name}_{ch+1:02}' for name in extras for ch in range(12)]
    features = np.concatenate(blocks)
    assert features.shape[1] == len(names) and np.isfinite(features).all()
    return features, names


In [ ]:
def fit_emg_classifier(training_features, training_labels):
    classifier = make_pipeline(StandardScaler(),
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))
    with threadpool_limits(limits=2):
        classifier.fit(training_features[:, :72], training_labels)
    assert np.array_equal(classifier.classes_, np.arange(17))
    return classifier


## 8. CNN fit and evaluation (same architecture and checkpoint rule)

In [ ]:
def fit_diagnostic_cnn(training, validation, output, subject):
    seed = 42 + 1009*subject
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    mean, std = fit_window_normalizer(training)
    training.mean = validation.mean = mean
    training.std = validation.std = std
    np.savez_compressed(output/'normalizer.npz', mean=mean, std=std)
    model = OriginalMultiKernelAttention1DCNN(48, 17, Config.DROPOUT)
    trainer = DiagnosticTrainer(model, output/'best_model.pt', 17)
    training_loader = DataLoader(training, batch_size=128, shuffle=True,
                                generator=torch.Generator().manual_seed(seed))
    validation_loader = DataLoader(validation, batch_size=128, shuffle=False)
    trainer.fit(training_loader, validation_loader)
    learning_report(trainer, output)
    model.load_state_dict(torch.load(output/'best_model.pt', map_location=Config.DEVICE, weights_only=True))
    json_write(output/'fit_manifest.json', dict(subject=subject, window_ms=training.window_ms,
        training_stride_ms=training.stride_ms, acc_centering=training.center_acc, seed=seed,
        train_windows=len(training), validation_windows=len(validation),
        selected_epoch=trainer.best_epoch, epochs_run=len(trainer.epoch_rows),
        optimizer_steps=len(training_loader)*len(trainer.epoch_rows),
        parameters=sum(parameter.numel() for parameter in model.parameters()),
        no_refit=True, window_selection=False, gate_thresholds=[.70, .80]))
    return model


In [ ]:
def predict_with_diagnostics(model, dataset, retain_attention=False):
    """Evaluate the checkpoint; collect embeddings and optionally attention on every window."""
    model.eval()
    captured, probabilities, embeddings, attention = {}, [], [], {}
    def capture(name):
        def hook(module, args, result):
            captured[name] = result.detach()
        return hook
    handles = [model.temporal_pool.register_forward_hook(capture('embedding'))]
    if retain_attention:
        handles += [model.temporal_pool.score.register_forward_hook(capture('time_scores'))]
        for stage in [1, 2, 3]:
            handles.append(getattr(model, f'attn{stage}').attention.register_forward_hook(capture(f'channel_stage{stage}')))
    try:
        with torch.no_grad():
            for signal, _ in DataLoader(dataset, batch_size=128, shuffle=False):
                logits = model(signal.to(Config.DEVICE))
                probabilities.append(logits.softmax(1).cpu().numpy())
                embeddings.append(captured['embedding'].cpu().numpy())
                if retain_attention:
                    for name in ['time_scores', 'channel_stage1', 'channel_stage2', 'channel_stage3']:
                        values = captured[name].softmax(-1) if name == 'time_scores' else captured[name]
                        attention.setdefault(name, []).append(values.squeeze(1 if name == 'time_scores' else -1).cpu().numpy())
    finally:
        for handle in handles:
            handle.remove()
    return np.concatenate(probabilities), np.concatenate(embeddings), {k:np.concatenate(v) for k,v in attention.items()}


In [ ]:
def evaluation_masks(metadata, window_ms, training_stride):
    elapsed = metadata.endpoint_ms.to_numpy() - (100+window_ms)
    return dict(primary=(elapsed % training_stride == 0),
                common=(metadata.endpoint_ms.to_numpy() >= 700) &
                       ((metadata.endpoint_ms.to_numpy()-700) % 200 == 0),
                dense=np.ones(len(metadata), dtype=bool))


In [ ]:
def save_predictions(dataset, cnn_probability, emg_probability, embeddings, attention,
                     features, feature_names, output, training_stride, split):
    output.mkdir(parents=True, exist_ok=True)
    assert np.isfinite(cnn_probability).all() and np.isfinite(emg_probability).all()
    assert np.allclose(cnn_probability.sum(1), 1, atol=1e-5)
    switch = (cnn_probability.max(1) < Config.GATE_CNN_THRESHOLD) & (emg_probability.max(1) > Config.GATE_EMG_THRESHOLD)
    gate_probability = np.where(switch[:, None], emg_probability, cnn_probability)
    frame = dataset.meta.copy()
    arms = {'cnn':cnn_probability, 'emg':emg_probability, 'gate':gate_probability}
    for arm, probability in arms.items():
        frame[f'{arm}_prediction'] = probability.argmax(1)+1
        frame[f'{arm}_confidence'] = probability.max(1)
        frame[f'{arm}_correct'] = probability.argmax(1) == dataset.y
        frame[f'{arm}_entropy'] = -(probability*np.log(np.maximum(probability, 1e-12))).sum(1)
    ordered = np.sort(cnn_probability, axis=1)
    frame['cnn_top2_margin'] = ordered[:, -1]-ordered[:, -2]
    frame['switch_to_emg'] = switch
    frame['recovered'] = ~frame.cnn_correct & frame.gate_correct
    frame['harmed'] = frame.cnn_correct & ~frame.gate_correct
    frame['both_wrong'] = ~frame.cnn_correct & ~frame.emg_correct
    masks = evaluation_masks(frame, dataset.window_ms, training_stride) if split == 'test' else {'primary':np.ones(len(frame), bool)}
    for name, mask in masks.items():
        frame[f'evaluate_{name}'] = mask
    frame.to_csv(output/'predictions.csv', index=False)
    np.savez_compressed(output/'probabilities.npz', y_true=dataset.y, **arms)
    np.savez_compressed(output/'features_embeddings.npz', features=features,
                        feature_names=np.asarray(feature_names), embeddings=embeddings)
    if attention:
        np.savez_compressed(output/'attention.npz', **attention)
    rows, gestures = [], []
    for grid, mask in masks.items():
        for arm, probability in arms.items():
            metrics, calibration = probability_metrics(dataset.y[mask], probability[mask])
            rows.append(dict(grid=grid, arm=arm, **metrics))
            pd.DataFrame(calibration).to_csv(output/f'calibration_{grid}_{arm}.csv', index=False)
            pd.DataFrame(confusion_matrix(dataset.y[mask], probability[mask].argmax(1), labels=range(17)),
                         index=range(1,18), columns=range(1,18)).to_csv(output/f'confusion_{grid}_{arm}.csv')
        for gesture, part in frame[mask].groupby('gesture'):
            wrong = part[~part.cnn_correct].cnn_prediction
            gestures.append(dict(grid=grid, gesture=gesture, windows=len(part),
                cnn_correct=int(part.cnn_correct.sum()), cnn_wrong=int((~part.cnn_correct).sum()),
                cnn_error_percent=100*float((~part.cnn_correct).mean()),
                gate_correct=int(part.gate_correct.sum()), gate_wrong=int((~part.gate_correct).sum()),
                gate_error_percent=100*float((~part.gate_correct).mean()),
                recovered=int(part.recovered.sum()), harmed=int(part.harmed.sum()),
                both_wrong=int(part.both_wrong.sum()),
                most_common_wrong_gesture=int(wrong.mode().iloc[0]) if len(wrong) else None))
    pd.DataFrame(rows).to_csv(output/'metrics.csv', index=False)
    pd.DataFrame(gestures).to_csv(output/'gesture_errors.csv', index=False)
    if split == 'test':
        primary = pd.DataFrame(gestures).query("grid == 'primary'")
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for ax, arm in zip(axes, ['cnn', 'gate']):
            ax.bar(primary.gesture, primary[f'{arm}_correct'], label='Correct', color='#25867a')
            ax.bar(primary.gesture, primary[f'{arm}_wrong'], bottom=primary[f'{arm}_correct'], label='Wrong', color='#ce604c')
            ax.set(title=f'{arm.upper()}: test windows per gesture', xlabel='Gesture', ylabel='Window count', xticks=range(1,18))
            ax.legend()
        save_figure(fig, output/'gesture_error_bars.png')
    return frame


## 9. Failure diagnosis: comparisons within the same subject, gesture and phase

In [ ]:
def feature_success_failure_comparison(frame, features, names, training_features, output):
    """Descriptive effects, not significance tests on correlated overlapping windows."""
    rows = []
    train_scale = training_features.std(0)+1e-12
    primary = frame[frame.evaluate_primary]
    for phase in ['all', 'early', 'middle', 'late']:
        selected = primary if phase == 'all' else primary[primary.phase == phase]
        for gesture, part in selected.groupby('gesture'):
            good_ids = part.index[part.cnn_correct].to_numpy()
            bad_ids = part.index[~part.cnn_correct].to_numpy()
            if not len(good_ids) or not len(bad_ids):
                continue
            good, bad = features[good_ids], features[bad_ids]
            for column, name in enumerate(names):
                good_median, bad_median = np.median(good[:, column]), np.median(bad[:, column])
                rows.append(dict(gesture=gesture, phase=phase, feature=name,
                    correct_windows=len(good_ids), wrong_windows=len(bad_ids),
                    correct_mean=float(good[:, column].mean()), wrong_mean=float(bad[:, column].mean()),
                    correct_median=float(good_median), wrong_median=float(bad_median),
                    median_difference_training_sd=float((bad_median-good_median)/train_scale[column])))
    pd.DataFrame(rows, columns=['gesture','phase','feature','correct_windows','wrong_windows',
        'correct_mean','wrong_mean','correct_median','wrong_median','median_difference_training_sd']).to_csv(output/'feature_correct_wrong.csv', index=False)


In [ ]:
def nearest_training_examples(training_values, test_values, train_meta, test_frame, space):
    """Distances use training-fitted scaling; true-label queries are diagnostic only."""
    scaler = StandardScaler().fit(training_values)
    train = scaler.transform(training_values)
    test = scaler.transform(test_values)
    all_distances, all_indices = [], []
    for gesture in range(1, 18):
        ids = np.flatnonzero(train_meta.gesture.to_numpy() == gesture)
        nearest = NearestNeighbors(n_neighbors=1, algorithm='brute', n_jobs=1).fit(train[ids])
        distances, neighbors = nearest.kneighbors(test)
        all_distances.append(distances[:, 0])
        all_indices.append(ids[neighbors[:, 0]])
    distances = np.stack(all_distances, axis=1)
    indices = np.stack(all_indices, axis=1)
    row_ids = np.arange(len(test))
    true = test_frame.gesture.to_numpy()-1
    predicted = test_frame.cnn_prediction.to_numpy()-1
    nearest_class = distances.argmin(1)
    nearest_ids = indices[row_ids, nearest_class]
    result = test_frame[['subject','gesture','native_repetition','window_start','window_end','cnn_correct','evaluate_primary','phase']].copy()
    result['space'] = space
    result['true_gesture_distance'] = distances[row_ids, true]
    result['cnn_predicted_gesture_distance'] = distances[row_ids, predicted]
    result['nearest_training_gesture'] = nearest_class+1
    result['nearest_training_row'] = nearest_ids
    result['nearest_training_repetition'] = train_meta.iloc[nearest_ids].native_repetition.to_numpy()
    result['nearest_training_window_start'] = train_meta.iloc[nearest_ids].window_start.to_numpy()
    other = distances.copy()
    other[row_ids, true] = np.inf
    result['nearest_other_distance'] = other.min(1)
    result['true_vs_other_distance_margin'] = result.nearest_other_distance-result.true_gesture_distance
    return result


In [ ]:
def error_timing_and_stride(frame, window_ms, output):
    phase_rows, streak_rows, stride_rows = [], [], []
    primary = frame[frame.evaluate_primary]
    for (gesture, phase), part in primary.groupby(['gesture', 'phase']):
        phase_rows.append(dict(gesture=gesture, phase=phase, windows=len(part),
            cnn_wrong=int((~part.cnn_correct).sum()), gate_wrong=int((~part.gate_correct).sum()),
            both_wrong=int(part.both_wrong.sum())))
    for (gesture, repetition), part in primary.groupby(['gesture','native_repetition']):
        part = part.sort_values('window_end')
        longest = current = 0
        for wrong in (~part.cnn_correct).to_numpy():
            current = current+1 if wrong else 0
            longest = max(longest, current)
        step_ms = float(np.diff(part.endpoint_ms).min()) if len(part)>1 else 0
        streak_rows.append(dict(gesture=gesture, repetition=repetition, windows=len(part),
            cnn_error_fraction=float((~part.cnn_correct).mean()), both_wrong_fraction=float(part.both_wrong.mean()),
            longest_error_streak_windows=longest, longest_error_endpoint_span_ms=max(0,longest-1)*step_ms,
            all_cnn_wrong=bool((~part.cnn_correct).all())))
    for stride in [50,100,200]:
        for offset in range(0,stride,50):
            mask = ((frame.endpoint_ms-(100+window_ms)-offset) % stride == 0)
            part = frame[mask]
            stride_rows.append(dict(output_stride_ms=stride, offset_ms=offset, windows=len(part),
                cnn_accuracy=float(part.cnn_correct.mean()), gate_accuracy=float(part.gate_correct.mean()),
                note='Same fitted model; output subsampling only'))
    pd.DataFrame(phase_rows).to_csv(output/'phase_errors.csv', index=False)
    pd.DataFrame(streak_rows).to_csv(output/'repetition_failures.csv', index=False)
    pd.DataFrame(stride_rows).to_csv(output/'output_stride_offsets.csv', index=False)


In [ ]:
def export_waveform_cases(dataset, frame, output):
    """One correct and one wrong primary-grid example per gesture, selected by confidence."""
    signals, raw, stimuli, labels, rows = [], [], [], [], []
    for gesture, part in frame[frame.evaluate_primary].groupby('gesture'):
        for correct in [True, False]:
            candidates = part[part.cnn_correct == correct].sort_values(['cnn_confidence','window_start'], ascending=[False,True])
            if candidates.empty:
                continue
            index = int(candidates.index[0])
            start = int(dataset.starts[index])
            signals.append(dataset.physical(index))
            raw.append(dataset.recording['raw_emg'][start:start+dataset.samples].T)
            stimuli.append(dataset.recording['stimulus'][start:start+dataset.samples])
            labels.append(dataset.recording['restimulus'][start:start+dataset.samples])
            rows.append(dict(case=len(rows), test_row=index, **frame.iloc[index].to_dict()))
    np.savez_compressed(output/'waveform_cases.npz', filtered_emg_and_acc=np.stack(signals),
                        raw_emg=np.stack(raw), stimulus=np.stack(stimuli), restimulus=np.stack(labels))
    pd.DataFrame(rows).to_csv(output/'waveform_case_index.csv', index=False)


## 10. Complete experiment for one subject, one window and one training stride

In [ ]:
def run_subject_configuration(recording, subject, window, stride, output):
    datasets = {split:WindowDataset(recording, split, window,
                Config.EVALUATION_GRID_MS if split == 'test' else stride)
                for split in ['train','validation','test']}
    features = {}
    for split, dataset in datasets.items():
        features[split], feature_names = diagnostic_features(dataset)
    classifier = fit_emg_classifier(features['train'], datasets['train'].y)
    joblib.dump(classifier, output/'emg_classifier.joblib')
    emg_probability = {split:classifier.predict_proba(values[:, :72]) for split,values in features.items()}
    variant_frames = {}
    for variant, centered in [('original_acc', False), ('centered_acc', True)]:
        folder = output/variant
        folder.mkdir()
        for dataset in datasets.values():
            dataset.center_acc = centered
        model = fit_diagnostic_cnn(datasets['train'], datasets['validation'], folder, subject)
        datasets['test'].mean = datasets['train'].mean
        datasets['test'].std = datasets['train'].std
        frames, embeddings = {}, {}
        for split, dataset in datasets.items():
            probability, embeddings[split], attention = predict_with_diagnostics(model, dataset, retain_attention=split=='test')
            frames[split] = save_predictions(dataset, probability, emg_probability[split], embeddings[split],
                attention, features[split], feature_names, folder/split, stride, split)
        diagnostic_dir = folder/'test'
        test_frame = frames['test']
        feature_success_failure_comparison(test_frame, features['test'], feature_names, features['train'], diagnostic_dir)
        with threadpool_limits(limits=2):
            physical = nearest_training_examples(features['train'], features['test'], datasets['train'].meta, test_frame, 'physical_features')
            learned = nearest_training_examples(embeddings['train'], embeddings['test'], datasets['train'].meta, test_frame, 'cnn_embedding')
        pd.concat([physical, learned]).to_csv(diagnostic_dir/'training_similarity.csv', index=False)
        error_timing_and_stride(test_frame, window, diagnostic_dir)
        export_waveform_cases(datasets['test'], test_frame, diagnostic_dir)
        activation_audit(model, datasets, folder)
        variant_frames[variant] = test_frame
        del model, embeddings
        gc.collect()
        torch.cuda.empty_cache()
    reference, candidate = variant_frames['original_acc'], variant_frames['centered_acc']
    assert np.array_equal(reference.window_end, candidate.window_end)
    paired = candidate[['subject','gesture','native_repetition','window_start','window_end','evaluate_primary','evaluate_common']].copy()
    paired['centering_recovered'] = ~reference.cnn_correct & candidate.cnn_correct
    paired['centering_harmed'] = reference.cnn_correct & ~candidate.cnn_correct
    paired.to_csv(output/'centering_paired_recovery.csv', index=False)


In [ ]:
def summarize_diagnostic_job(root):
    metrics, gestures = [], []
    for manifest_path in root.glob('w*/s*/S*/*/fit_manifest.json'):
        folder = manifest_path.parent
        fit = json.loads(manifest_path.read_text())
        for split in ['train','validation','test']:
            for name, destination in [('metrics.csv', metrics), ('gesture_errors.csv', gestures)]:
                if not (folder/split/name).exists():
                    continue  # Preserve useful partial outputs if a later diagnostic fails.
                data = pd.read_csv(folder/split/name)
                data['subject'] = fit['subject']
                data['window_ms'] = fit['window_ms']
                data['training_stride_ms'] = fit['training_stride_ms']
                data['variant'] = folder.name
                data['split'] = split
                destination.append(data)
    if not metrics:
        return
    metrics = pd.concat(metrics, ignore_index=True)
    metrics.to_csv(root/'all_subject_metrics.csv', index=False)
    metrics.groupby(['window_ms','training_stride_ms','variant','split','grid','arm'])[['accuracy','f1_macro']].mean().to_csv(root/'mean_subject_metrics.csv')
    if not gestures:
        return
    errors = pd.concat(gestures, ignore_index=True)
    errors.to_csv(root/'all_gesture_errors.csv', index=False)
    for (window,stride,variant), part in errors.query("split == 'test' and grid == 'primary'").groupby(['window_ms','training_stride_ms','variant']):
        matrix = part.pivot(index='subject',columns='gesture',values='cnn_error_percent')
        fig, ax = plt.subplots(figsize=(12, max(3, len(matrix)*.28)))
        sns.heatmap(matrix, ax=ax, cmap='Reds', vmin=0, vmax=100, annot=True, fmt='.0f', cbar_kws={'label':'CNN test error %'})
        ax.set_title(f'{window} ms window / {stride} ms training stride / {variant}')
        save_figure(fig, root/f'errors_w{window}_s{stride}_{variant}.png')


In [ ]:
def run_diagnostic_experiments():
    assert Config.DEVICE.type == 'cuda', 'Enable a Kaggle GPU.'
    assert Config.EXERCISE_IDS == (1,) and Config.GESTURE_MIN == 1 and Config.GESTURE_MAX == 17
    assert not Config.REFIT_ON_TRAIN_PLUS_VAL and not Config.AUGMENT_TRAIN
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    root = Config.KAGGLE_WORKING/f'db7_diagnostic_study_{stamp}'
    root.mkdir(parents=True)
    subjects = [1] if Config.SMOKE else list(range(1,23))
    if Config.SMOKE:
        Config.MIN_EPOCHS, Config.MAX_EPOCHS, Config.PATIENCE = 1, 2, 2
    manifest = dict(automation=Config.AUTOMATION, subjects=subjects, smoke=Config.SMOKE,
        windows_ms=Config.WINDOWS_MS, strides_ms=Config.STRIDES_MS, labels=list(range(1,18)),
        protocol='Within-subject 4 train / 1 validation / 1 test repetition; no refit',
        window_selection=False, checkpoint_selection='Minimum validation loss within each fit',
        native_grid='First endpoint = 100 ms trim + window length; same stride for train and validation. Test primary matches training stride.',
        dense_test_grid_ms=50, common_test_grid='Endpoints 700 ms + multiples of 200 ms in every configuration',
        gate='CNN confidence <0.70 AND EMG confidence >0.80; frozen; EMG StandardScaler + shrinkage LDA on 72 features',
        feature_diagnostics='300 physical features; no significance tests treating overlapping windows as independent',
        case_selection='Highest-confidence correct and wrong example per gesture; examples not representative random samples',
        limits='One seed. Existing test data explored before; future changes require independent confirmation. Rest excluded. Zero-phase per-repetition filtering is not online causal. Internal S18 identity and physical channel semantics unresolved. Flat/repeated extrema are indicators, not proven ADC clipping.',
        versions=dict(python=sys.version,torch=torch.__version__,numpy=np.__version__,pandas=pd.__version__))
    json_write(root/'run_manifest.json', manifest)
    completed = []
    try:
        for window in Config.WINDOWS_MS:
            for stride in Config.STRIDES_MS:
                assert window in [200,400,600] and stride in [50,100,200]
                for subject in subjects:
                    folder = root/f'w{window}'/f's{stride}'/f'S{subject:02}'
                    folder.mkdir(parents=True)
                    print(f'RUN: window={window} stride={stride} subject={subject}', flush=True)
                    recording = load_diagnostic_subject(subject, folder)
                    run_subject_configuration(recording, subject, window, stride, folder)
                    completed.append(dict(subject=subject,window_ms=window,stride_ms=stride))
                    json_write(root/'progress.json', completed)
                    del recording
                    gc.collect()
                summarize_diagnostic_job(root)
        expected = len(subjects)*len(Config.WINDOWS_MS)*len(Config.STRIDES_MS)*2
        assert len(list(root.glob('w*/s*/S*/*/fit_manifest.json'))) == expected
        json_write(root/'completion.json',dict(success=True,cnn_fits=expected,completed=completed,smoke=Config.SMOKE))
    except Exception:
        (root/'FAILURE.txt').write_text(traceback.format_exc(),encoding='utf-8')
        raise
    finally:
        try:
            summarize_diagnostic_job(root)
        except Exception:
            (root/'SUMMARY_FAILURE.txt').write_text(traceback.format_exc(), encoding='utf-8')
        finally:
            print('OUTPUT ZIP:',shutil.make_archive(str(root),'zip',root_dir=root),flush=True)
    return root


## Run the configured experiment

GitHub inserts its per-job overrides immediately above this cell. All preceding cells define code only, apart from imports/configuration. Full jobs retain the complete epoch budget.

In [ ]:
RESULTS_DIRECTORY = run_diagnostic_experiments()
print(RESULTS_DIRECTORY)
